In [ ]:
!cat /etc/os-release | head -2

PRETTY_NAME="Ubuntu 22.04.5 LTS"
NAME="Ubuntu"


In [ ]:
!whoami

root


In [ ]:
!ps aux | wc -l

19


In [ ]:
import torch, subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
p = torch.cuda.get_device_properties(0)
print(f"{p.name} | compute capability sm_{p.major}{p.minor} | {p.total_memory/1e9:.2f} GB")
print("torch", torch.__version__)

Tesla T4, 15360 MiB, 580.82.07
Tesla T4 | compute capability sm_75 | 15.64 GB
torch 2.11.0+cu128


In [ ]:
import time, torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "ibm-granite/granite-4.1-3b"     # pick from aidc.nadir.sh/library, T4 16GB

# everyone answers this one, so we can compare
SHARED_PROMPT = "In one sentence, what is a data centre for?"
# and one of your own
YOUR_PROMPT   = "Explain why GPUs are useful for AI in simple terms"

tok = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.float16, device_map="cuda",
    trust_remote_code=True)                                # some models ship their own modelling code

def ask(prompt, max_new_tokens=120):
    ids = tok(prompt, return_tensors="pt").to("cuda")
    t0 = time.time()
    out = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False)
    dt = time.time() - t0
    answer = tok.decode(out[0][ids["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
    n = out.shape[-1] - ids["input_ids"].shape[-1]
    print(f"\n--- {prompt}\n{answer}\n({n} tokens in {dt:.1f}s, {n/dt:.1f} tok/s)")
    return answer

print("model:", MODEL)
ask(SHARED_PROMPT)
ask(YOUR_PROMPT)

Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

model: ibm-granite/granite-4.1-3b

--- In one sentence, what is a data centre for?
A data centre is a facility used to house computer systems and associated components, such as telecommunications and storage systems. It provides an environment optimized for data storage, processing, and distribution, ensuring high availability, security, and efficient operation of IT infrastructure.
(51 tokens in 3.0s, 17.1 tok/s)

--- Explain why GPUs are useful for AI in simple terms
.

GPUs are like super-fast helpers for computers. They can do many calculations at the same time, which is perfect for AI tasks like training big models or running complex calculations. This makes AI projects faster and more efficient.
(46 tokens in 2.5s, 18.4 tok/s)


'.\n\nGPUs are like super-fast helpers for computers. They can do many calculations at the same time, which is perfect for AI tasks like training big models or running complex calculations. This makes AI projects faster and more efficient.'

## do_sample=True

In [ ]:

def ask(prompt, max_new_tokens=120):
    ids = tok(prompt, return_tensors="pt").to("cuda")
    t0 = time.time()
    out = model.generate(**ids, max_new_tokens=120, do_sample=True, temperature=0.8)
    dt = time.time() - t0
    answer = tok.decode(out[0][ids["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
    n = out.shape[-1] - ids["input_ids"].shape[-1]
    print(f"\n--- {prompt}\n{answer}\n({n} tokens in {dt:.1f}s, {n/dt:.1f} tok/s)")
    return answer

print("model:", MODEL)
ask(SHARED_PROMPT)
ask(YOUR_PROMPT)

model: ibm-granite/granite-4.1-3b

--- In one sentence, what is a data centre for?
A data centre is a facility used to house, manage, and monitor an organization's critical computing, storage, and networking equipment, providing a secure environment for data storage, processing, and distribution.
(41 tokens in 4.1s, 10.0 tok/s)

--- Explain why GPUs are useful for AI in simple terms
.
GPU stands for Graphics Processing Unit. GPUs are specialized processors designed to handle complex calculations and graphical tasks efficiently. They are particularly good at parallel processing, meaning they can perform many calculations at the same time. AI, especially deep learning, involves a lot of mathematical calculations and data processing. GPUs can accelerate these tasks significantly compared to traditional CPUs (Central Processing Units). This speed and efficiency make GPUs ideal for training large AI models and running inference (using the trained models to make predictions or decisions). In

'.\nGPU stands for Graphics Processing Unit. GPUs are specialized processors designed to handle complex calculations and graphical tasks efficiently. They are particularly good at parallel processing, meaning they can perform many calculations at the same time. AI, especially deep learning, involves a lot of mathematical calculations and data processing. GPUs can accelerate these tasks significantly compared to traditional CPUs (Central Processing Units). This speed and efficiency make GPUs ideal for training large AI models and running inference (using the trained models to make predictions or decisions). In simple terms, GPUs help AI work faster and more efficiently by doing many calculations simultaneously, which is essential for'